# Transaction Foundation Model on Ray — Part 1: Setup

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

### Anyscale Technical Demo — Ray Data + Ray Train + Ray Serve

---

## Business context

Banks and fintechs are converging on **transaction foundation models** (TFMs): a single self-supervised transformer pretrained on raw transaction sequences that emits a reusable **customer embedding**, which downstream models (fraud, churn, credit, personalization) consume instead of dozens of hand-built feature pipelines. Stripe, Visa (TREASURE), Nubank, and Revolut (PRAGMA) have all published variants of this recipe.

The model itself is small and is *not* the hard part. The hard parts are **engineering at scale**: tokenizing petabytes of transactions, pretraining across many GPUs, and re-embedding every customer on a schedule — then serving those embeddings both in batch and in real time. That is exactly the Ray/Anyscale story, and what this series builds.

Our one modeling upgrade over the standard NVIDIA blueprint is a **static/dynamic field split** in the tokenizer and model (the idea behind Visa TREASURE and FATA-Trans) — cheaper and a stronger inductive bias than flattening every field into the token stream. More on that in Part 2.

## Architecture

```
                       ┌─────────────────────── Anyscale ───────────────────────┐
 raw transactions ──►  │ [Ray Data]   static/dynamic tokenization (map_groups)   │
   (Parquet, S3)       │ [Ray Train]  masked-feature pretraining (PyTorch + DDP) │
                       │ [Ray Data]   batch embedding extraction (CPU read+GPU)  │
                       │ [XGBoost]    downstream fraud: raw vs FM vs fusion       │
                       │ [Ray Serve]  online embedding + fraud score (cached)     │
                       └─────────────────────────────────────────────────────────┘
```

Every stage is the **same code** from laptop to multi-node cluster — you change `ScalingConfig`, not your program. The whole series runs at **`smoke` scale** (a few thousand cards, a 2-layer model) so it completes in minutes on a small cluster; flip the scale to `small`/`full` and `use_gpu=True` for the real distributed story.

## The series

**This is a series.** We build a transaction foundation model end-to-end on Ray — the same code from a single node to a multi-node GPU cluster:

| # | Notebook | Ray primitive |
|---|----------|---------------|
| **1** | **Setup** *(this notebook)* | — |
| 2 | Load & explore the data | Ray Data |
| 3 | Tokenize — the static/dynamic split | Ray Data `map_groups` |
| 4 | Pretrain — masked-feature modeling | Ray Train (DDP/FSDP) |
| 5 | Batch embedding extraction | Ray Data `map_batches` |
| 6 | Downstream fraud — raw vs FM vs fusion | XGBoost |
| 7 | Online serving | Ray Serve |

---

## Step 1: Connect to the Ray cluster

In an Anyscale Workspace, Ray is **pre-initialized** — no cluster setup, no Spark context. The cell below installs the template's dependencies; on the GPU base image PyTorch is already present, so the list is light: `ray`, `torch`, `xgboost`, and a few supporting libraries.

> In production you'd install from the generated `python_depset.lock`. Here we install from `requirements.txt` for portability.

In [ ]:
!pip install -q -r requirements.txt

`ray.init()` **attaches** to the workspace's running cluster (it does not start a new one). `working_dir` ships this template's `src/` package to every worker, so the same imports resolve on the head node and on remote tasks/actors.

In [ ]:
import sys, os

# Make the template's `src/` package importable from the notebook.
DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import ray

# In an Anyscale Workspace, Ray is already running — this attaches to it.
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.utils import print_cluster_resources
print_cluster_resources()

You should see the head node and its resources. On a fresh or idle cluster the head node intentionally schedules no work — GPU/CPU **worker** nodes autoscale up on demand when later stages request them, and scale back to zero when idle. You don't manage that; Ray does.

### Per-scale configuration

Everything that varies between runs lives in one file per scale under `configs/` (`smoke.yaml` / `small.yaml` / `full.yaml`) — data sampling, model dims, training, and embedding settings. Diffing two files shows the complete difference between two scales; there is no hidden inheritance. The notebooks default to `smoke`; you change a single `SCALE` variable to scale up.

---

## Next

**Part 2 — Load & explore the data**: download the IBM TabFormer benchmark with Ray Data, inspect the schema, and see *in the data* why the static/dynamic split, log-bucketed amounts, time-aware positions, and a temporal eval split are the right modeling choices.